# App 1 — Segmentation Training Run 1
**Model:** YOLOv11s-seg (COCO pretrained)  
**Dataset:** app1-seg-splits (591 images, nc=3, 70/15/15 stratified split)  
**Classes:** corner_breakout | spalling_body | spalling_rail_seat  
**Oversampling:** spalling_rail_seat 2× in train only  
**Ground2Tech Engineering — Parallel segmentation branch**

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install ultralytics --quiet

In [ ]:
# ── Verify dataset structure ──────────────────────────────────────────────────
import os
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/app1-seg-splits')

for split in ('train', 'val', 'test'):
    n_img = len(list((DATASET_ROOT / split / 'images').glob('*')))
    n_lbl = len(list((DATASET_ROOT / split / 'labels').glob('*.txt')))
    print(f'{split:6} images={n_img}  labels={n_lbl}  match={n_img==n_lbl}')

In [ ]:
# ── Write dataset.yaml to working dir ────────────────────────────────────────
import yaml

cfg = {
    'path':  '/kaggle/input/app1-seg-splits',
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    3,
    'names': ['corner_breakout', 'spalling_body', 'spalling_rail_seat']
}

yaml_path = '/kaggle/working/dataset_seg.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(open(yaml_path).read())

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
# Rationale for hyperparameters:
#   lr0=0.008 / lrf=0.001: same schedule as detection Run 3 which converged
#     cleanly at epoch 117. Lower than default (0.01) to account for small dataset.
#   patience=40: allows recovery from local plateaus without wasting GPU time.
#   mosaic=0.8: standard for small datasets — creates multi-scale context.
#   flipud=0.3 / fliplr=0.5: appropriate for ground-level track photos
#     (vertical flip less common but physically plausible, horizontal common).
#   degrees=15: slight rotational augmentation for handheld camera variation.
#   NO mixup, NO copy_paste: Run 6 showed these destabilise small datasets
#     by creating unrealistic composite images. Removed deliberately.
#   batch=16: T4 16GB VRAM handles this at imgsz=640 comfortably.

from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')

results = model.train(
    data      = '/kaggle/working/dataset_seg.yaml',
    epochs    = 150,
    batch     = 16,
    patience  = 40,
    imgsz     = 640,
    device    = 0,
    optimizer = 'SGD',
    lr0       = 0.008,
    lrf       = 0.001,
    mosaic    = 0.8,
    flipud    = 0.3,
    fliplr    = 0.5,
    degrees   = 15,
    hsv_h     = 0.015,
    hsv_s     = 0.5,
    hsv_v     = 0.4,
    mixup     = 0.0,
    copy_paste= 0.0,
    name      = 'app1_seg_run1_yolo11s',
    project   = '/kaggle/working/runs',
    plots     = True,
    save      = True,
)

In [ ]:
# ── Evaluate on val ───────────────────────────────────────────────────────────
metrics = model.val()
print('\n=== SEGMENTATION METRICS (val) ===')
print(f'box mAP50:      {metrics.box.map50:.4f}')
print(f'box mAP50-95:   {metrics.box.map:.4f}')
print(f'seg mAP50:      {metrics.seg.map50:.4f}')   # PRIMARY decision gate
print(f'seg mAP50-95:   {metrics.seg.map:.4f}')
print()
print('Per-class seg mAP50:')
names = ['corner_breakout', 'spalling_body', 'spalling_rail_seat']
for i, name in enumerate(names):
    try:
        print(f'  [{i}] {name:<24} {metrics.seg.maps[i]:.4f}')
    except:
        print(f'  [{i}] {name:<24} n/a')

print('\n=== DECISION GATE ===')
seg_map50 = metrics.seg.map50
if seg_map50 > 0.70:
    print(f'✓ seg mAP50={seg_map50:.3f} > 0.70 → PROCEED to ONNX export + Flutter rewrite')
elif seg_map50 >= 0.60:
    print(f'? seg mAP50={seg_map50:.3f} in [0.60, 0.70] → COMPARE with detection Run 3 (0.702)')
    print('  Consider: is mask quality worth the Flutter inference rewrite complexity?')
else:
    print(f'✗ seg mAP50={seg_map50:.3f} < 0.60 → DISCARD branch, document failure, keep Run 3')

In [ ]:
# ── ONNX export (run only if seg mAP50 > 0.65) ───────────────────────────────
# Output will have TWO tensors:
#   output0: [1, 7, 8400]    — box coords + nc=3 class scores
#   output1: [1, 32, 160, 160] — prototype masks

model.export(
    format   = 'onnx',
    opset    = 12,
    simplify = True,
    dynamic  = False,
    imgsz    = 640,
)

# Verify output tensor shapes
import onnxruntime as ort
best_onnx = list(Path('/kaggle/working/runs/app1_seg_run1_yolo11s/weights').glob('best.onnx'))
if best_onnx:
    sess = ort.InferenceSession(str(best_onnx[0]))
    print('ONNX output tensors:')
    for o in sess.get_outputs():
        print(f'  {o.name}  {o.shape}')
    # Expected:
    #   output0  [1, 7, 8400]
    #   output1  [1, 32, 160, 160]

In [ ]:
# ── Save artefacts ────────────────────────────────────────────────────────────
import shutil

weights_dir = Path('/kaggle/working/runs/app1_seg_run1_yolo11s/weights')
output_dir  = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

for f in weights_dir.glob('best*'):
    shutil.copy(f, output_dir / f.name)
    print(f'Saved: {f.name}')

# Copy training plots
plots_dir = Path('/kaggle/working/runs/app1_seg_run1_yolo11s')
for plot in plots_dir.glob('*.png'):
    shutil.copy(plot, output_dir / plot.name)

print(f'\nOutput files:')
for f in sorted(output_dir.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')